In [36]:
%pip install pyserial
import serial
import time
%pip install websocket-client[optional]

import json

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
def parse_mania_beatmap(filepath):
    hits = []
    in_hitobjects = False

    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # wait until we reach the [HitObjects] section
            if line == '[HitObjects]':
                in_hitobjects = True
                continue

            # once we' in [HitObjects), parse each note
            if in_hitobjects and line:
                parts = line.split(',')

                # x position tdecides teh lane 
                x = int(parts[0]) - 64
                lane = x // 128

                time_ms = int(parts[2])

                hits.append((time_ms, lane))

    # sort by time so notes are in chronological order
    return sorted(hits)


# load the beatmap 
hits = parse_mania_beatmap(r'C:\mania_hackathon\950937 Sarahkuai - 4K Patterns Training Pack Vol1\Sarahkuai - 4K Patterns Training Pack Vol.1 (Sarahkuai) [DOUBLE BASIC].osu')

# hits = parse_mania_beatmap(r'C:\mania_hackathon\950937 Sarahkuai - 4K Patterns Training Pack Vol1\Sarahkuai - 4K Patterns Training Pack Vol.1 (Sarahkuai) [SINGLE BASIC].osu')

print(f"Loaded {len(hits)} notes")
print("First 5:", hits[:5])

Loaded 320 notes
First 5: [(9600, 0), (9600, 1), (9750, 0), (9750, 1), (10050, 0)]


Arduino connected


In [86]:
ser.close()

In [ ]:
try:
    ser.close()
except:
    pass

ser = serial.Serial('COM4', 115200)
time.sleep(3)

# how many ms before the note to send the command (negative = send later)
PREFIRE_MS = -275

# how long to hold the servo in pressed position before releasing
HOLD_MS = 40

# notes within this many ms of each other count as the same chord
CHORD_WINDOW_MS = 5

# index of the next note to fire
hit_index = 0

print("Waiting for you to start the map in osu...")

# check tosu until the map is playing and near the beginning
while True:
    data = requests.get("http://localhost:24050/json/v2").json()
    state = data['state']['name']
    current_ms = data['beatmap']['time']['live']
    if state == 'play' and current_ms < 500:
        break

# take 5 timing samples and keep the one with the fastest request time
# faster request = more accurate sync between wall clock and game clock
best_sync = None
for _ in range(5):
    wall_before = time.perf_counter()
    data = requests.get("http://localhost:24050/json/v2").json()
    wall_after = time.perf_counter()
    game_ms = data['beatmap']['time']['live']
    request_time_ms = (wall_after - wall_before) * 1000

    if best_sync is None or request_time_ms < best_sync[2]:
        # store: game time, midpoint wall time, request duration
        best_sync = (game_ms, wall_before + (wall_after - wall_before) / 2, request_time_ms)

sync_game_ms = best_sync[0]   # what time the game was at when we synced
sync_wall    = best_sync[1]   # what our wall clock was at that moment
sync_quality = best_sync[2]   # how long the request took (lower = better)
print(f"Synced at game time {sync_game_ms}ms, sync quality {sync_quality:.1f}ms")

# main scheduler loop - no more HTTP calls in here, uses local clock only
while hit_index < len(hits):

    # calculate current game time using wall clock since sync point
    elapsed_ms = (time.perf_counter() - sync_wall) * 1000
    current_ms = sync_game_ms + elapsed_ms

    # fire any notes that are due
    while hit_index < len(hits) and hits[hit_index][0] - PREFIRE_MS <= current_ms:

        # get the time of this note
        base_time = hits[hit_index][0]

        # build a chord byte by combining all notes within CHORD_WINDOW_MS
        chord_byte = 0
        while hit_index < len(hits) and hits[hit_index][0] - base_time <= CHORD_WINDOW_MS:
            _, lane = hits[hit_index]
            chord_byte |= (1 << lane)  # set the bit for this lane
            hit_index += 1

        # how early or late to fire
        offset = current_ms - base_time
        print(f"FIRING {bin(chord_byte)} at {current_ms:.0f}ms (target {base_time}ms) offset {offset:.0f}ms")

        # send the chord byte via serial port to arduino so it will move the right servos
        ser.write(bytes([chord_byte]))

        # wait for servo to complete the press
        time.sleep(HOLD_MS / 1000.0)

        # send 0 to release all servos back to neutral
        ser.write(bytes([0]))



Arduino connected
Waiting for you to start the map in osu...
Synced at game time 5132ms, sync quality 2038.3ms
FIRING 0b11 at 10242ms (target 9600ms) offset 642ms
FIRING 0b11 at 10242ms (target 9750ms) offset 492ms
FIRING 0b11 at 10333ms (target 10050ms) offset 283ms
FIRING 0b11 at 10450ms (target 10200ms) offset 250ms
FIRING 0b1100 at 10750ms (target 10500ms) offset 250ms
FIRING 0b1100 at 10900ms (target 10650ms) offset 250ms
FIRING 0b1100 at 11200ms (target 10950ms) offset 250ms
FIRING 0b1100 at 11350ms (target 11100ms) offset 250ms
FIRING 0b11 at 11650ms (target 11400ms) offset 250ms
FIRING 0b1100 at 11950ms (target 11700ms) offset 250ms
FIRING 0b11 at 12250ms (target 12000ms) offset 250ms
FIRING 0b11 at 12400ms (target 12150ms) offset 250ms
FIRING 0b11 at 12700ms (target 12450ms) offset 250ms
FIRING 0b11 at 12850ms (target 12600ms) offset 250ms
FIRING 0b1100 at 13150ms (target 12900ms) offset 250ms
FIRING 0b1100 at 13300ms (target 13050ms) offset 250ms
FIRING 0b1100 at 13600ms (tar

KeyboardInterrupt: 

In [ ]:
data = requests.get("http://localhost:24050/json/v2").json()
print("state:", data['state']['name'])
print("current_ms:", data['beatmap']['time']['live'])
print("first hit:", hits[0])
print("first hit target:", hits[0][0] - PREFIRE_MS)
print("condition:", hits[0][0] - PREFIRE_MS <= data['beatmap']['time']['live'])

state: play
current_ms: 421
first hit: (9600, 1)
first hit target: 9450
condition: False
